# Preparación de radiografías: train, validación y test

Este notebook contiene **toda la implementación en sus celdas**. No importa módulos locales `.py` ni necesita ejecutarlos. Usar **Run All / Ejecutar todo**, de arriba hacia abajo, con el kernel `Python (iaproyecto)`.

Por defecto procesa imágenes sintéticas para comprobar la ejecución. Para radiografías reales, cambiar `DATASET`, las rutas y el hallazgo en la configuración. No entrena un modelo.

Si falta instalar el entorno: desde la raíz del repositorio ejecutar `.\.venv\Scripts\python.exe -m pip install -e ".[notebooks]"`.

El recorrido sigue el PLAN y los ejemplos del profesor. Los adaptadores, el control por paciente y la exportación son implementación propia del proyecto.


In [ ]:
from pathlib import Path
import os
import json
from datetime import datetime, timezone
from collections import Counter
from html import escape

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

%matplotlib inline

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'pyproject.toml').is_file() and (p / 'notebooks').is_dir()), Path.cwd())
os.chdir(PROJECT_ROOT)
print('Carpeta de trabajo:', PROJECT_ROOT)

def tabla(rows, limit=12):
    rows = list(rows)[:limit]
    if not rows:
        print('Sin registros para mostrar.')
        return
    fields = list(dict.fromkeys(k for row in rows for k in row))
    head = ''.join('<th>' + escape(str(k)) + '</th>' for k in fields)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(row.get(k, ''))) + '</td>' for k in fields) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))


## 1. Configurar el experimento

Elegir **un** formato: `demo`, `generic`, `chexpert` o `nih`. Los hallazgos de ejemplo son configurables.

- **Genérico:** CSV con `image_path,patient_id,target`; `view` es necesario si se activa el filtro frontal.
- **CheXpert:** CSV con `Path`, columna del hallazgo y `Frontal/Lateral`. `image_root` se une literalmente con `Path`. Solo se aceptan etiquetas explícitas 0 y 1; -1 y vacío se excluyen y registran. El ejemplo subdivide `train.csv`: el test resultante es interno, no el oficial.
- **NIH:** `Data_Entry_2017.csv`, imágenes y, si están disponibles, `train_val_list.txt` y `test_list.txt` junto al CSV. El test oficial se conserva y validación sale de train.
- **Particiones explícitas:** columna `split` completa con `train,val,test`, o `train,test` para obtener validación dentro de train.

Se admiten PNG/JPEG/TIFF de un frame, grises de 8/16 bits y RGB de 8 bits. DICOM requiere otro lector. Cada ejecución completa crea otra carpeta de salida; los originales no se sobrescriben.


In [ ]:
DATASET = 'demo'  # Cambiar a 'generic', 'chexpert' o 'nih' para radiografías reales.

opciones = {
    'demo': {'dataset': 'generic', 'target': 'target',
             'metadata': 'data/raw/demo_notebook/metadata.csv',
             'image_root': 'data/raw/demo_notebook', 'image_size': 32, 'frontal_only': True},
    'generic': {'dataset': 'generic', 'target': 'target',
                'metadata': 'data/raw/mis_radiografias/metadata.csv',
                'image_root': 'data/raw/mis_radiografias', 'image_size': 224, 'frontal_only': False},
    'chexpert': {'dataset': 'chexpert', 'target': 'Pleural Effusion',
                 'metadata': 'data/raw/CheXpert-v1.0-small/train.csv',
                 'image_root': 'data/raw', 'image_size': 224, 'frontal_only': True},
    'nih': {'dataset': 'nih', 'target': 'Effusion',
            'metadata': 'data/raw/nih/Data_Entry_2017.csv',
            'image_root': 'data/raw/nih', 'image_size': 224, 'frontal_only': True},
}
if DATASET not in opciones:
    raise ValueError('Elegir uno de los cuatro formatos disponibles.')

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%f')
config_input = dict(opciones[DATASET])
config_input.update(seed=42, ratios=[0.70, 0.15, 0.15], normalization='unit', split_attempts=1000,
                    output=f'data/processed/{DATASET}_notebook_{RUN_ID}')
# Ajustar aquí metadata, image_root, target o normalization='zscore' según el experimento.
display(config_input)


## 2. Demo sintética opcional

Esta función se usa solo en modo `demo`. Genera 120 imágenes de ruido de 60 pacientes ficticios. **No son radiografías ni permiten medir rendimiento clínico.** Con datos reales no genera ni descarga archivos.


In [ ]:
"""Genera imágenes geométricas SINTÉTICAS para probar la ejecución, no la IA."""

import argparse

import csv

from pathlib import Path

import numpy as np

from PIL import Image

def generate(output, patients=60):
    output = Path(output)
    if output.exists():
        raise ValueError(f'La carpeta ya existe: {output}')
    output.mkdir(parents=True)
    (output / 'images').mkdir()
    rng = np.random.default_rng(123)
    records = []
    for patient in range(patients):
        for study in range(2):
            image = rng.integers(0, 256, size=(48, 40), dtype=np.uint8)
            path = f'images/p{patient:03d}_{study}.png'
            Image.fromarray(image).save(output / path)
            records.append({'image_path': path, 'patient_id': f'p{patient:03d}',
                            'target': patient % 2, 'view': 'PA'})
    with (output / 'metadata.csv').open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)
    (output / 'SINTETICO.txt').write_text('Datos artificiales de prueba. No son radiografías.\n', encoding='utf-8')
    return output


In [ ]:
if DATASET == 'demo':
    synthetic_root = PROJECT_ROOT / config_input['image_root']
    if not synthetic_root.exists():
        generate(synthetic_root)
    if not (synthetic_root / 'SINTETICO.txt').is_file():
        raise ValueError('La carpeta de demo existente no tiene el marcador SINTETICO.txt.')
    print('Demo sintética disponible:', synthetic_root)
else:
    print('Se usarán los archivos reales existentes; no se descargan automáticamente.')


## 3. Leer y transformar imágenes

Escala fija uint8/255 o uint16/65535; no se divide por el máximo de cada imagen. Se convierte a grises, se redimensiona sin deformar ni recortar y se completa con bordes negros. La resolución debe verificarse para el hallazgo elegido. El hash detecta píxeles exactos en grises, no todas las variantes recomprimidas.


In [ ]:
"""Lectura de PNG/JPEG y TIFF de un solo frame, sin recorte anatómico."""

import hashlib

import warnings

from pathlib import Path

import numpy as np

from PIL import Image, ImageOps

def read_image(path: Path):
    """Devuelve grises float32 [0,1], hash de píxeles y características originales.

    La escala depende del formato de almacenamiento, nunca del máximo de cada
    imagen. No interpreta DICOM, windowing, MONOCHROME1 ni metadatos clínicos.
    """
    with warnings.catch_warnings():
        warnings.simplefilter("error", Image.DecompressionBombWarning)
        with Image.open(path) as original:
            if original.format not in {"PNG", "JPEG", "TIFF"}:
                raise ValueError("Formato no admitido: usar PNG, JPEG o TIFF")
            if getattr(original, "n_frames", 1) != 1:
                raise ValueError("Imagen multiframe no admitida")
            image = ImageOps.exif_transpose(original)
            mode = image.mode
            if mode == "RGB":
                pixels = np.asarray(image.convert("L"))
                scale = 255.0
            elif mode == "L":
                pixels = np.asarray(image)
                scale = 255.0
            elif mode.startswith("I;16") or (mode == "I" and original.format == "PNG"):
                pixels = np.asarray(image)
                if pixels.min() < 0 or pixels.max() > 65535:
                    raise ValueError("Valores fuera del rango unsigned de 16 bits")
                scale = 65535.0
            else:
                raise ValueError(f"Modo {mode} no admitido; convertir de forma explícita")
            fingerprint = hashlib.sha256()
            fingerprint.update(str((pixels.shape, scale)).encode())
            fingerprint.update(np.asarray(pixels, dtype="<u2").tobytes())
            info = {"width": image.width, "height": image.height, "mode": mode,
                    "intensity_scale": int(scale), "pixel_hash": fingerprint.hexdigest()}
            return pixels.astype(np.float32) / scale, info

def transform(path: Path, size: int, expected_hash=None):
    """Redimensiona sin deformar y centra con padding negro. Salida (H,W)."""
    pixels, info = read_image(path)
    if expected_hash is not None and info['pixel_hash'] != expected_hash:
        raise ValueError(f"La imagen cambió después de la auditoría: {path}")
    height, width = pixels.shape
    factor = size / max(height, width)
    new_size = (max(1, round(width * factor)), max(1, round(height * factor)))
    resized = np.asarray(Image.fromarray(pixels).resize(new_size, Image.Resampling.BILINEAR))
    result = np.zeros((size, size), dtype=np.float32)
    x, y = (size - new_size[0]) // 2, (size - new_size[1]) // 2
    result[y:y + new_size[1], x:x + new_size[0]] = resized
    return result


## 4. Lectores y auditoría

Los adaptadores convierten los formatos a registros comunes. Se distinguen archivos ausentes de ilegibles y se registran etiquetas excluidas. Se conserva una sola copia de un duplicado compatible; si las copias tienen etiquetas, pacientes o splits incompatibles, se excluyen todas. No se imputan etiquetas ni se eliminan casos solo por ser raros.


In [ ]:
"""Adaptadores explícitos para etiquetas binarias y auditoría de archivos."""

import csv

import re

from collections import defaultdict

from pathlib import Path

def read_metadata(config):
    source = Path(config["metadata"]).resolve()
    image_root = Path(config["image_root"]).resolve()
    dataset = config["dataset"]
    target = config["target"]
    with source.open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        fields = set(reader.fieldnames or [])
        required = {"image_path", "patient_id", target}
        if dataset == "chexpert":
            required = {"Path", target}
        elif dataset == "nih":
            required = {"Image Index", "Patient ID", "Finding Labels"}
        if config.get("frontal_only"):
            required.add({"chexpert": "Frontal/Lateral", "nih": "View Position",
                          "generic": "view"}[dataset])
        if not required <= fields:
            raise ValueError(f"Faltan columnas en {source.name}: {sorted(required - fields)}")
        raw = list(reader)
    if not raw:
        raise ValueError("El CSV no tiene registros")
    if dataset == "nih":
        vocabulary = {label.strip() for row in raw for label in row['Finding Labels'].split('|')}
        if target not in vocabulary or target == "No Finding":
            raise ValueError("El target NIH debe ser un hallazgo presente en Finding Labels")
        # NIH distribuye las imágenes en varios subdirectorios con el mismo nombre 'images'.
        index = defaultdict(list)
        for path in image_root.rglob('*'):
            if path.is_file() and path.suffix.lower() in {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}:
                index[path.name].append(path.resolve())
        official = {}
        lists = [source.parent / 'train_val_list.txt', source.parent / 'test_list.txt']
        if any(p.exists() for p in lists):
            if not all(p.is_file() for p in lists):
                raise ValueError('NIH: deben estar presentes train_val_list.txt y test_list.txt juntos')
            if any(row.get('split', '').strip() for row in raw):
                raise ValueError('NIH: elegir listas oficiales o columna split, no ambas')
            for split, path in zip(['train', 'test'], lists):
                for name in path.read_text(encoding='utf-8-sig').splitlines():
                    name = name.strip()
                    if not name:
                        continue
                    if name in official:
                        raise ValueError(f'NIH: imagen repetida en listas oficiales: {name}')
                    official[name] = split
            if set(official.values()) != {'train', 'test'}:
                raise ValueError('NIH: las dos listas oficiales deben contener imágenes')
    records, exclusions = [], []
    for number, row in enumerate(raw, start=2):
        item = {"source_row": number, "image_path": "", "patient_id": "",
                "target_original": "", "split": row.get("split", "").strip().lower()}
        if item['split'] == 'valid':
            item['split'] = 'val'
        reason = None
        if dataset == "nih":
            name = row['Image Index'].strip()
            matches = index.get(name, [])
            item.update(image_path=name, patient_id=row['Patient ID'].strip(),
                        target_original=row['Finding Labels'])
            if official:
                item['split'] = official.get(name, '')
                if not item['split']:
                    reason = 'missing_official_split'
            labels = {label.strip() for label in row['Finding Labels'].split('|')}
            if not labels or '' in labels or ('No Finding' in labels and len(labels) > 1):
                reason = reason or 'invalid_label'
            elif len(matches) != 1:
                reason = reason or ('missing_file' if not matches else 'ambiguous_filename')
            else:
                item['image_path'] = str(matches[0])
            item['target'] = int(target in labels)
            view = row.get('View Position', '').strip().upper()
            frontal = view in {'AP', 'PA'}
        else:
            relative = row['Path' if dataset == 'chexpert' else 'image_path'].strip().replace('\\', '/')
            item['image_path'] = relative
            if not relative:
                reason = 'missing_path'
            resolved = (image_root / relative).resolve()
            if not resolved.is_relative_to(image_root):
                reason = 'path_outside_image_root'
            item['image_path'] = str(resolved)
            if dataset == 'chexpert':
                match = re.search(r'(?:^|/)(patient\d+)(?:/|$)', relative)
                item['patient_id'] = match.group(1) if match else ''
                frontal = row.get('Frontal/Lateral', '').strip().lower() == 'frontal'
            else:
                item['patient_id'] = row['patient_id'].strip()
                frontal = row.get('view', '').strip().upper() in {'AP', 'PA', 'FRONTAL'}
            label = row[target].strip()
            item['target_original'] = label
            if label in {'0', '0.0', '1', '1.0'}:
                item['target'] = int(float(label))
            else:
                reason = reason or ({'': 'missing_label', '-1': 'uncertain_label',
                                     '-1.0': 'uncertain_label'}.get(label, 'invalid_label'))
        if not item['patient_id']:
            reason = reason or 'missing_patient_id'
        if config.get('frontal_only') and not frontal:
            reason = reason or 'non_frontal_view'
        if reason:
            exclusions.append(dict(item, reason=reason))
        else:
            records.append(item)
    # Una fila con split suministrado indica una partición oficial/explícita.
    # No se mezcla silenciosamente con filas sin asignación.
    supplied = [r.get('split', '').strip().lower() for r in raw]
    if any(supplied) and (not all(supplied) or any(s not in {'train','val','valid','test'} for s in supplied)):
        raise ValueError("La columna split debe estar completa y contener train, val/valid o test")
    return records, exclusions, len(raw)

def audit(records, exclusions):
    by_hash = defaultdict(list)
    for record in records:
        path = Path(record['image_path'])
        try:
            if not path.is_file():
                raise FileNotFoundError(path)
            _, info = read_image(path)
            by_hash[info['pixel_hash']].append(dict(record, **info))
        except (OSError, ValueError, Warning) as error:
            exclusions.append(dict(record, reason='missing_file' if isinstance(error, FileNotFoundError)
                                   else 'unreadable_image', detail=str(error)))
    kept = []
    for copies in by_hash.values():
        if len({r['target'] for r in copies}) > 1:
            reason = 'duplicate_conflicting_label'
        elif len({r['patient_id'] for r in copies}) > 1:
            reason = 'duplicate_across_patients'
        elif len({r['split'] for r in copies}) > 1:
            reason = 'duplicate_across_splits'
        else:
            kept.append(copies[0])
            copies = copies[1:]
            reason = 'duplicate_image'
        exclusions.extend(dict(r, reason=reason) for r in copies)
    return sorted(kept, key=lambda r: (r['patient_id'], r['image_path']))


## 5. Separación por paciente

Todas las imágenes de una persona permanecen juntas. La búsqueda aproxima las proporciones de imágenes y clases con semilla fija, sin usar resultados de modelos. Se conservan las particiones explícitas. Debe haber positivos y negativos en cada conjunto; para una evaluación útil harán falta suficientes pacientes de ambas clases.


In [ ]:
"""Split reproducible por paciente con búsqueda de proporciones aproximadas."""

import numpy as np

NAMES = ('train', 'val', 'test')

def validate(splits):
    patients, hashes = set(), set()
    for name in NAMES:
        rows = splits[name]
        if {r['target'] for r in rows} != {0, 1}:
            raise ValueError(f"{name} necesita ejemplos de ambas clases tras la auditoría")
        current = {r['patient_id'] for r in rows}
        fingerprints = {r['pixel_hash'] for r in rows}
        if patients & current or hashes & fingerprints:
            raise ValueError("Fuga entre particiones: pacientes o imágenes repetidas")
        patients.update(current)
        hashes.update(fingerprints)

def partition(records, config, names, ratios):
    patients = sorted({r['patient_id'] for r in records})
    if len(patients) < 2 * len(names):
        raise ValueError(f"Se necesitan al menos {2*len(names)} pacientes y ambas clases en cada partición")
    ratios = np.array(ratios, dtype=float)
    sizes = [max(1, round(len(patients) * ratio)) for ratio in ratios[1:]]
    sizes.insert(0, len(patients) - sum(sizes))
    if sizes[0] < 1:
        raise ValueError("Las proporciones no dejan pacientes en train")
    lookup = {p: i for i, p in enumerate(patients)}
    counts = np.zeros((len(patients), 2), dtype=int)
    for row in records:
        counts[lookup[row['patient_id']], row['target']] += 1
    if np.any(counts.sum(axis=0) < len(names)):
        raise ValueError("No hay suficientes imágenes de cada clase para las particiones")
    rng = np.random.default_rng(config['seed'])
    best, best_score = None, float('inf')
    # Selección únicamente por tamaño y etiquetas, nunca por rendimiento del modelo.
    for _ in range(config.get('split_attempts', 1000)):
        order = rng.permutation(len(patients))
        groups = np.split(order, np.cumsum(sizes)[:-1])
        totals = np.array([counts[group].sum(axis=0) for group in groups])
        if np.any(totals == 0):
            continue
        score = float(np.square(totals / counts.sum(axis=0) - ratios[:, None]).sum())
        if score < best_score:
            best_score, best = score, groups
    if best is None:
        raise ValueError("No se encontró split con ambas clases. Revisar pacientes/clases o aumentar split_attempts")
    assignment = {patients[int(i)]: name for name, group in zip(names, best) for i in group}
    splits = {name: [dict(r, split=name) for r in records if assignment[r['patient_id']] == name]
              for name in names}
    return splits

def make_splits(records, config):
    supplied = {r['split'] for r in records if r['split']}
    if supplied:
        splits = {name: [r for r in records if r['split'] == name] for name in NAMES}
        method = 'explicit_three_way_split'
        if supplied == {'train', 'test'}:
            if {r['patient_id'] for r in splits['train']} & {r['patient_id'] for r in splits['test']}:
                raise ValueError('Fuga entre train y test oficiales: pacientes repetidos')
            a, b, _ = config['ratios']
            internal = partition(splits['train'], config, ('train', 'val'), [a/(a+b), b/(a+b)])
            splits.update(internal)
            method = 'explicit_test_patient_validation_from_train'
        validate(splits)
        return splits, method
    splits = partition(records, config, NAMES, config['ratios'])
    validate(splits)
    return splits, 'seeded_patient_search'


## 6. Validación y exportación

Estas funciones contienen el resto del flujo. La exportación usa arrays mapeados en disco. Media y desviación se calculan **solo con train**. Con `unit` se mantiene [0,1]; con `zscore` se aplican las estadísticas de train a los tres conjuntos. No se balancea, aumenta ni ajusta con test.


In [ ]:
__version__ = "0.1.0"

"""CLI: rx-preparar --config configs/chexpert.json."""

import argparse

import csv

import hashlib

import importlib.metadata

import json

import shutil

import sys

import tempfile

from collections import Counter

from datetime import datetime, timezone

from pathlib import Path

import numpy as np

DEFAULTS = {'image_size': 224, 'seed': 42, 'ratios': [0.7, 0.15, 0.15],
            'normalization': 'unit', 'frontal_only': False, 'split_attempts': 1000}

def load_config(path):
    with Path(path).open(encoding='utf-8-sig') as stream:
        given = json.load(stream)
    required = {'dataset', 'target', 'metadata', 'image_root', 'output'}
    allowed = required | set(DEFAULTS)
    if not required <= given.keys() or given.keys() - allowed:
        raise ValueError(f"Configuración: faltan {sorted(required-given.keys())}; "
                         f"claves desconocidas {sorted(given.keys()-allowed)}")
    config = DEFAULTS | given
    if config['dataset'] not in {'generic', 'nih', 'chexpert'}:
        raise ValueError('dataset debe ser generic, nih o chexpert')
    if not isinstance(config['target'], str) or not config['target'].strip():
        raise ValueError('target debe ser un nombre no vacío')
    for name in ['image_size', 'split_attempts', 'seed']:
        if type(config[name]) is not int or config[name] < (0 if name == 'seed' else 1):
            raise ValueError(f'{name} debe ser un entero válido')
    if not 8 <= config['image_size'] <= 2048:
        raise ValueError('image_size debe estar entre 8 y 2048')
    ratios = np.asarray(config['ratios'], dtype=float)
    if ratios.shape != (3,) or not np.isfinite(ratios).all() or np.any(ratios <= 0) or not np.isclose(ratios.sum(), 1):
        raise ValueError('ratios debe contener tres proporciones positivas que sumen 1')
    if type(config['frontal_only']) is not bool:
        raise ValueError('frontal_only debe ser true o false')
    if config['normalization'] not in {'unit', 'zscore'}:
        raise ValueError('normalization debe ser unit o zscore')
    for key in ['metadata', 'image_root', 'output']:
        config[key] = str(Path(config[key]).resolve())
    if not Path(config['metadata']).is_file() or not Path(config['image_root']).is_dir():
        raise ValueError('Faltan el CSV o la carpeta de imágenes; revisar metadata e image_root')
    output = Path(config['output'])
    inputs = [Path(config['image_root']), Path(config['metadata']).parent]
    if any(output == p or output.is_relative_to(p) or p.is_relative_to(output) for p in inputs):
        raise ValueError('output debe estar separado de las carpetas de entrada')
    if output.exists():
        raise ValueError(f'La salida ya existe: {output}. Usar otra ruta para conservar el experimento')
    return config

def write_csv(path, records, fields=None):
    fields = fields or sorted({key for row in records for key in row}) or ['source_row', 'reason']
    with path.open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(records)

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


def export_prepared(config, records, exclusions, source_count, splits, method):
    output = Path(config['output'])
    if output.exists():
        raise ValueError(f'La salida ya existe: {output}')
    output.parent.mkdir(parents=True, exist_ok=True)
    stage = Path(tempfile.mkdtemp(prefix=output.name + '.preparing-', dir=output.parent)).resolve()
    try:
        size = config['image_size']
        estimated_bytes = sum(len(rows) for rows in splits.values()) * size * size * 4
        if shutil.disk_usage(stage).free < estimated_bytes * 1.1:
            raise ValueError(f'Espacio insuficiente; arrays necesitan aproximadamente {estimated_bytes/2**30:.2f} GiB')
        total, squared, count = 0.0, 0.0, 0
        print('Calculando estadísticas solo sobre train...', flush=True)
        for row in splits['train']:
            image = transform(Path(row['image_path']), size, row['pixel_hash']).astype(np.float64)
            total += float(image.sum())
            squared += float(np.square(image).sum())
            count += image.size
        mean = total / count
        std = float(np.sqrt(max(0.0, squared / count - mean * mean)))
        if config['normalization'] == 'zscore' and std < 1e-8:
            raise ValueError('Train tiene varianza nula; no se puede estandarizar')
        stats = {'normalization': config['normalization'], 'train_mean': mean, 'train_std': std,
                 'statistics_source': 'train_only_after_resize_and_padding', 'image_size': size,
                 'grayscale': True, 'resize': 'bilinear_preserve_aspect_black_padding',
                 'unit_scale': 'uint8/255 or uint16/65535', 'shape': 'N,H,W', 'dtype': 'float32'}
        report = {'created_at_utc': datetime.now(timezone.utc).isoformat(),
                  'project_version': __version__, 'python': sys.version,
                  'dependencies': {p: importlib.metadata.version(p) for p in ['numpy', 'Pillow']},
                  'metadata_sha256': hashlib.sha256(Path(config['metadata']).read_bytes()).hexdigest(),
                  'source_rows': source_count, 'accepted_images': len(records),
                  'excluded_rows': len(exclusions), 'exclusion_reasons': dict(Counter(r['reason'] for r in exclusions)),
                  'split_method': method, 'seed': config['seed'], 'splits': {},
                  'checks': {'patients_disjoint': True, 'exact_pixels_disjoint': True,
                             'both_classes_each_split': True},
                  'limitations': ['Exact duplicate detection only; near duplicates require further review.',
                                 'No automatic clinical quality or label correctness verification.',
                                 'No resampling, augmentation, outlier deletion or label imputation.']}
        for name in NAMES:
            rows = splits[name]
            array = np.lib.format.open_memmap(stage / f'X_{name}.npy', mode='w+', dtype=np.float32,
                                               shape=(len(rows), size, size))
            try:
                for i, row in enumerate(rows):
                    image = transform(Path(row['image_path']), size, row['pixel_hash'])
                    if config['normalization'] == 'zscore':
                        image = (image - mean) / std
                    array[i] = image
                array.flush()
            finally:
                # Cerrar el mapping también en Windows antes de renombrar/eliminar.
                array._mmap.close()
                del array
            np.save(stage / f'y_{name}.npy', np.array([r['target'] for r in rows], dtype=np.int64), allow_pickle=False)
            exported = [dict(r, array_index=i) for i, r in enumerate(rows)]
            write_csv(stage / f'{name}.csv', exported)
            report['splits'][name] = {'images': len(rows), 'patients': len({r['patient_id'] for r in rows}),
                                     'class_0': sum(r['target'] == 0 for r in rows),
                                     'class_1': sum(r['target'] == 1 for r in rows),
                                     'image_fraction': len(rows) / len(records)}
            print(f'{name}: {len(rows)} imágenes exportadas', flush=True)
        assert len(records) + len(exclusions) == source_count
        write_csv(stage / 'exclusiones.csv', exclusions)
        write_json(stage / 'config.json', config)
        write_json(stage / 'preprocessing.json', stats)
        write_json(stage / 'report.json', report)
        write_json(stage / 'labels.json', {'target': config['target'], '0': 'negative_annotation',
                                         '1': 'positive_annotation',
                                         'note': 'Negativo para el hallazgo no significa radiografía normal.'})
        stage.rename(output)
    finally:
        if stage.exists():
            if stage.parent != output.parent.resolve() or not stage.name.startswith(output.name + '.preparing-'):
                raise RuntimeError('Ruta temporal inesperada; no se elimina')
            shutil.rmtree(stage)
    print(f'Preparación terminada: {output}', flush=True)
    return output

def prepare(config_path):
    config = load_config(config_path)
    records, exclusions, source_count = read_metadata(config)
    print(f'Metadatos: {source_count} filas. Auditando {len(records)} imágenes...', flush=True)
    records = audit(records, exclusions)
    output = Path(config['output'])
    try:
        splits, method = make_splits(records, config)
    except ValueError:
        # Un diagnóstico separado permite revisar exclusiones aun si no hay un split viable.
        diagnostic = output.with_name(output.name + '.audit.csv')
        diagnostic.parent.mkdir(parents=True, exist_ok=True)
        if not diagnostic.exists():
            write_csv(diagnostic, exclusions)
        print(f'No se exportaron conjuntos. Diagnóstico de auditoría: {diagnostic}', file=sys.stderr)
        raise
    return export_prepared(config, records, exclusions, source_count, splits, method)


## 7. Ejecutar lectura y auditoría

La configuración visible y el resumen de exclusiones se guardan en `work/notebooks/`, incluso si luego no se puede formar un split válido. Una descarga parcial se reflejará en las exclusiones; documentar esa limitación.


In [ ]:
work_dir = PROJECT_ROOT / 'work' / 'notebooks'
work_dir.mkdir(parents=True, exist_ok=True)
config_path = work_dir / f'config_{RUN_ID}.json'
write_json(config_path, config_input)
config = load_config(config_path)
records, exclusions, source_count = read_metadata(config)
records = audit(records, exclusions)
write_csv(work_dir / f'auditoria_{RUN_ID}.csv', exclusions)
assert len(records) + len(exclusions) == source_count
print('Filas de entrada:', source_count)
print('Imágenes aceptadas:', len(records), '| filas excluidas:', len(exclusions))
tabla([{'motivo': reason, 'cantidad': count} for reason, count in Counter(r['reason'] for r in exclusions).items()])
tabla([{k: r.get(k, '') for k in ('source_row', 'target_original', 'reason', 'detail')} for r in exclusions], limit=10)


## 8. Crear train, validación y test

El split precede al EDA para decisiones de modelado y al ajuste de transformaciones. Se muestran conteos para verificar la integridad. El análisis visual posterior utiliza solamente train.


In [ ]:
splits, split_method = make_splits(records, config)
validate(splits)
resumen = []
for name in NAMES:
    rows = splits[name]
    resumen.append({'conjunto': name, 'imágenes': len(rows),
                    'pacientes': len({r['patient_id'] for r in rows}),
                    'negativos': sum(r['target'] == 0 for r in rows),
                    'positivos': sum(r['target'] == 1 for r in rows)})
print('Método:', split_method)
tabla(resumen)
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(NAMES))
ax.bar(x-0.18, [r['negativos'] for r in resumen], width=0.36, label='Negativo para el hallazgo')
ax.bar(x+0.18, [r['positivos'] for r in resumen], width=0.36, label='Positivo para el hallazgo')
ax.set_xticks(x, NAMES)
ax.set_ylabel('Imágenes')
ax.set_title('Distribución de las particiones')
ax.legend()
plt.tight_layout()
plt.show()


## 9. EDA visual de train

Comparar originales y representación preparada: resolución, orientación y pérdida de detalle. No se excluye automáticamente una imagen por este gráfico. En demo se verá ruido artificial. El histograma corresponde a una muestra reproducible de train, después de resize y padding; no al dataset completo.


In [ ]:
rng = np.random.default_rng(config['seed'])
train_rows = splits['train']
selected = rng.choice(len(train_rows), size=min(3, len(train_rows)), replace=False)
fig, axes = plt.subplots(len(selected), 2, figsize=(8, 3 * len(selected)), squeeze=False)
for axes_row, index in zip(axes, selected):
    row = train_rows[int(index)]
    original, info = read_image(Path(row['image_path']))
    prepared = transform(Path(row['image_path']), config['image_size'], row['pixel_hash'])
    axes_row[0].imshow(original, cmap='gray', vmin=0, vmax=1)
    axes_row[0].set_title(f"Original {info['width']}×{info['height']} · etiqueta {row['target']}")
    axes_row[1].imshow(prepared, cmap='gray', vmin=0, vmax=1)
    axes_row[1].set_title(f"Preparada {prepared.shape[1]}×{prepared.shape[0]}")
    for ax in axes_row:
        ax.axis('off')
plt.tight_layout()
plt.show()

sample = rng.choice(len(train_rows), size=min(32, len(train_rows)), replace=False)
counts = np.zeros(50, dtype=np.int64)
for index in sample:
    row = train_rows[int(index)]
    image = transform(Path(row['image_path']), config['image_size'], row['pixel_hash'])
    hist, edges = np.histogram(image, bins=50, range=(0, 1))
    counts += hist
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(edges[:-1], counts, width=0.02, align='edge')
ax.set(xlabel='Intensidad escalada [0,1]', ylabel='Píxeles', title='Muestra de train tras resize y padding')
plt.tight_layout()
plt.show()


## 10. Exportar los conjuntos

A 224×224 float32 se requieren unos 196 KiB por imagen. Se comprueba espacio libre y no se sobrescriben experimentos. El directorio final aparece solo si termina la exportación. Para rehacerla, ejecutar todo desde el principio para generar otra salida.


In [ ]:
estimated_gib = len(records) * config['image_size'] ** 2 * 4 / 2**30
print(f'Espacio aproximado para arrays: {estimated_gib:.3f} GiB')
output_dir = export_prepared(config, records, exclusions, source_count, splits, split_method)
print('Archivos generados:')
for path in sorted(output_dir.iterdir()):
    print(' -', path.name)


## 11. Cargar y verificar los arrays

Las posiciones de `y` coinciden con `X` y con `array_index` del CSV. Los arrays se abren en modo lectura y mapeados en disco. Se verifican valores finitos en bloques para limitar la RAM. Pacientes, rutas y hashes son trazabilidad, no features predictoras.


In [ ]:
datasets = {}
for name in NAMES:
    images = np.load(output_dir / f'X_{name}.npy', mmap_mode='r', allow_pickle=False)
    labels = np.load(output_dir / f'y_{name}.npy', allow_pickle=False)
    assert images.shape == (len(splits[name]), config['image_size'], config['image_size'])
    assert len(labels) == len(images) and set(labels.tolist()) == {0, 1}
    assert labels.tolist() == [r['target'] for r in splits[name]]
    for start in range(0, len(images), 64):
        assert np.isfinite(images[start:start+64]).all()
    datasets[name] = (images, labels)
    print(f'{name}: X={images.shape}, {images.dtype}; y={labels.shape}, {labels.dtype}')
validate(splits)
X_train, y_train = datasets['train']
X_val, y_val = datasets['val']
X_test, y_test = datasets['test']
preprocessing = json.loads((output_dir / 'preprocessing.json').read_text(encoding='utf-8'))
print('Correcto: pacientes e imágenes independientes, clases y arrays válidos.')
display(preprocessing)


## 12. Listos para entrenar y testear

La vista vectorial sirve para un clasificador tabular. Para una CNN se puede agregar un canal. No se entrena un modelo en esta etapa.

- Ajustar el modelo solo con train.
- Elegir configuraciones con validación.
- Evaluar el modelo seleccionado con test al final.
- Balancear, si se justifica, únicamente train.
- Conservar los manifiestos y la configuración del experimento.


In [ ]:
X_train_vectores = X_train.reshape(len(X_train), -1)
X_val_vectores = X_val.reshape(len(X_val), -1)
X_test_vectores = X_test.reshape(len(X_test), -1)
print('Vectores:', X_train_vectores.shape, X_val_vectores.shape, X_test_vectores.shape)
# Para CNN con canal primero: X_train[:, None, :, :] -> N,1,H,W
print('Dataset listo en:', output_dir)
print('Datos sintéticos de prueba.' if DATASET == 'demo' else 'Revisar auditoría y muestra visual antes de entrenar.')


## Límites y referencias

Las comprobaciones no certifican calidad clínica, etiquetas correctas ni independencia de duplicados aproximados. Las exclusiones pueden sesgar la muestra; informar sus cantidades y motivos. En CheXpert, incierto no se convierte en negativo. En NIH, un negativo para el target significa que ese hallazgo no está anotado como positivo, no que la radiografía sea normal.

Base local: `PLAN_PROYECTO_RX_TORAX.md`, slides 05/06/09 y ejemplos del profesor de split, escalado, desbalance y EDA de imágenes.

Referencias: [CheXpert oficial](https://stanfordmlgroup.github.io/competitions/chexpert/) y [Pillow](https://pillow.readthedocs.io/en/stable/reference/ImageOps.html).

El notebook `02_pruebas_pipeline.ipynb` verifica la implementación de estas celdas sin importar el paquete local `.py`.
